# kaiming-uniform-sf-init — ex2: Kaiming uniform SF init for Conv2d weights (fan_in = IC * kH * kW)

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `kaiming-uniform-sf-init`. Running the final beacon cell reports progress against the `Init: Kaiming uniform SF init` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Init: Kaiming uniform SF init` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`kaiming-uniform-sf-init`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "kaiming-uniform-sf-init"
DD_SUBTOPIC = "Init: Kaiming uniform SF init"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Kaiming uniform SF init for Conv2d weights — quick refresher

For Conv2d, `fan_in` is NOT `in_channels` alone — it's the size of the **receptive patch** that produces one output unit:

```
fan_in = in_channels * kernel_h * kernel_w
sf     = 1 / sqrt(fan_in)
weight ~ Uniform(-sf, +sf)              shape (OC, IC, kH, kW)
```

Same `Uniform(-sf, +sf)` recipe as `nn.Linear`; only the `fan_in` formula changes. PyTorch's `nn.Conv2d.reset_parameters` uses exactly this.

**Exemplar.** `IC=3, kH=3, kW=3 -> fan_in = 27, sf = 1/sqrt(27) ~= 0.192`. Weights of shape `(OC, 3, 3, 3)` are sampled on `(-0.192, 0.192)`. Empirical `std ~= sf / sqrt(3) ~= 0.111`.

### Exercise 2 — Kaiming uniform SF init for Conv2d weights (fan_in = IC * kH * kW)

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the Conv2d-specific Kaiming-uniform recipe — `fan_in = in_channels * kernel_h * kernel_w`, `sf = 1/sqrt(fan_in)`, `weight ~ Uniform(-sf, +sf)` of shape `(out_channels, in_channels, kernel_h, kernel_w)`.
> Keywords: kaiming, uniform, init, conv2d, receptive-fan-in
> ```

**KCs targeted:** `kaiming-uniform-sf-init`, `conv-fan-in-formula`

Implement `kaiming_uniform_sf_conv2d(out_channels, in_channels, kernel_h, kernel_w, generator)`. The Conv2d weight initializer ARENA uses (and PyTorch's `nn.Conv2d` default):

1. `fan_in = in_channels * kernel_h * kernel_w` — NOT just `in_channels`. This is the size of the receptive PATCH that produces one output unit, not just the channel dimension.
2. `sf = 1 / sqrt(fan_in)`.
3. Sample `(out_channels, in_channels, kernel_h, kernel_w)` floats uniformly on `(-sf, +sf)` using `generator`.
4. Return as a `torch.Tensor` (4-D).

**Contrast with ex1.** Ex1's Linear init uses `fan_in = in_features`. For Conv2d the `fan_in` formula changes — the spatial extent of the kernel matters because every spatial position participates in the summation that produces one output activation. Same `Uniform(-sf, +sf)` recipe, different `fan_in`.

**Why this differs from `nn.Linear` viewed as 1x1 Conv.** A `Conv2d(IC, OC, kernel_size=1)` has `fan_in = IC * 1 * 1 = IC` — which matches `Linear(IC, OC)`. The two coincide at kernel size 1. They diverge for bigger kernels: a 3x3 conv has `fan_in = 9 * IC` — `sf` shrinks by `sqrt(9) = 3`.

Hint: `t.rand(shape, generator=g)` is uniform on `[0, 1)`. To get `(-sf, +sf)`, do `(t.rand(shape, generator=g) * 2 - 1) * sf`.

Output: `torch.Tensor` of shape `(out_channels, in_channels, kernel_h, kernel_w)`.

In [ ]:
def kaiming_uniform_sf_conv2d(
    out_channels: int, in_channels: int, kernel_h: int, kernel_w: int,
    generator: t.Generator,
) -> Tensor:
    """Sample Conv2d weight ~ Uniform(-1/sqrt(IC*kH*kW), +1/sqrt(IC*kH*kW))."""
    raise NotImplementedError()


def _test_ex2():
    import math

    # --- shape ---
    g = t.Generator().manual_seed(0)
    w = kaiming_uniform_sf_conv2d(4, 3, 3, 3, g)
    assert isinstance(w, t.Tensor), f'expected torch.Tensor, got {type(w).__name__}'
    assert w.shape == (4, 3, 3, 3), f'shape: {w.shape}'

    # --- bounds: |w| <= sf for every element ---
    fan_in = 3 * 3 * 3
    sf = 1.0 / math.sqrt(fan_in)
    assert w.abs().max().item() <= sf + 1e-6, (
        f'all entries in (-sf, +sf); max |w| = {w.abs().max().item()} vs sf = {sf:.4f}'
    )

    # --- large-sample empirical std ≈ sf / sqrt(3) ---
    g2 = t.Generator().manual_seed(1)
    OC, IC, kH, kW = 200, 16, 5, 5
    w_big = kaiming_uniform_sf_conv2d(OC, IC, kH, kW, g2)   # 200*16*25 = 80k samples
    fan_in_big = IC * kH * kW
    sf_big = 1.0 / math.sqrt(fan_in_big)
    assert w_big.abs().max().item() <= sf_big + 1e-6
    expected_std = sf_big / math.sqrt(3.0)
    empirical_std = w_big.std().item()
    rel_err = abs(empirical_std - expected_std) / expected_std
    assert rel_err < 0.05, (
        f'empirical std {empirical_std:.5f} too far from expected '
        f'{expected_std:.5f} (rel err {rel_err:.4f}); did you use fan_in = IC alone?'
    )

    # --- the key distinction from ex1: fan_in MUST include kH * kW ---
    # If init mistakenly used fan_in = IC, std would be sqrt(kH*kW) times too large.
    wrong_sf = 1.0 / math.sqrt(IC)
    wrong_std = wrong_sf / math.sqrt(3.0)
    assert empirical_std < wrong_std * 0.5, (
        f'empirical std {empirical_std} is close to the IC-only formula {wrong_std:.5f} — '
        'fan_in must include kernel_h * kernel_w'
    )

    # --- generator honored: same seed → same tensor ---
    g_a = t.Generator().manual_seed(42)
    g_b = t.Generator().manual_seed(42)
    w_a = kaiming_uniform_sf_conv2d(2, 3, 3, 3, g_a)
    w_b = kaiming_uniform_sf_conv2d(2, 3, 3, 3, g_b)
    assert t.allclose(w_a, w_b), 'same seed must produce the same weight tensor'

    # --- different seed → different tensor ---
    g_c = t.Generator().manual_seed(99)
    w_c = kaiming_uniform_sf_conv2d(2, 3, 3, 3, g_c)
    assert not t.allclose(w_a, w_c), 'different seed should produce different tensor'

    # --- kernel-size scaling: a 3x3 conv has sf ~ 3x smaller than a 1x1 conv at same IC ---
    g_1 = t.Generator().manual_seed(7)
    g_3 = t.Generator().manual_seed(7)
    w_1x1 = kaiming_uniform_sf_conv2d(100, 64, 1, 1, g_1)  # fan_in = 64
    w_3x3 = kaiming_uniform_sf_conv2d(100, 64, 3, 3, g_3)  # fan_in = 576
    ratio = w_3x3.abs().max().item() / w_1x1.abs().max().item()
    # sf_3x3 / sf_1x1 = sqrt(64/576) = sqrt(1/9) = 1/3.
    assert 0.25 < ratio < 0.45, (
        f'sf must scale as 1/sqrt(IC*kH*kW); 3x3 vs 1x1 max-abs ratio {ratio:.3f}, expected ~1/3'
    )

    # --- agreement with the Linear formula at kernel = 1x1 ---
    # For (kH, kW) = (1, 1), fan_in = IC, matching Linear(IC, OC). The sf should match.
    g_lin = t.Generator().manual_seed(123)
    g_1x1 = t.Generator().manual_seed(123)
    # Linear-equivalent: sf = 1/sqrt(IC)
    raw_lin = t.rand(64, 100, generator=g_lin)            # we mimic Linear init manually
    sf_lin  = 1.0 / math.sqrt(64)
    w_lin   = (raw_lin * 2 - 1) * sf_lin                  # shape (64, 100)
    w_conv1x1 = kaiming_uniform_sf_conv2d(100, 64, 1, 1, g_1x1)  # shape (100, 64, 1, 1)
    # Compare the underlying scalar distributions (max-abs and std).
    assert abs(w_lin.abs().max().item() - w_conv1x1.abs().max().item()) < 0.05, (
        'at kernel 1x1, Conv2d and Linear inits should have the same scale'
    )
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def kaiming_uniform_sf_conv2d(
    out_channels: int, in_channels: int, kernel_h: int, kernel_w: int,
    generator: t.Generator,
) -> Tensor:
    fan_in = in_channels * kernel_h * kernel_w
    sf = fan_in ** -0.5
    raw = t.rand(out_channels, in_channels, kernel_h, kernel_w, generator=generator)
    return (raw * 2 - 1) * sf
```

**Why `fan_in = IC * kH * kW`, not `IC`.** For a forward Conv2d, each output activation is the sum of `IC * kH * kW` weighted inputs. The Kaiming derivation argues that the input-output variance is preserved when `Var(w) * fan_in = 1` (for ReLU it's 2; for the SF form, it's whatever the constant works out to under `Uniform(-sf, sf)`). The relevant `fan_in` is the count of inputs AGGREGATED per output unit. For Conv2d that's `IC * kH * kW` — every spatial position of the kernel patch.

**At kernel 1x1, Conv2d == Linear.** `fan_in = IC * 1 * 1 = IC`. Both inits sample on the same `(-1/sqrt(IC), +1/sqrt(IC))` interval. This is why 1x1 convs are sometimes called 'pointwise linear layers' — they really are linear in the channel dimension, with no spatial context.

**Why bigger kernels → smaller weights.** A 3x3 conv at the same `IC` has `9x` more inputs feeding each output, so each individual weight should be `3x` smaller on average to preserve the pre-activation scale. The test asserts the 1x1 vs 3x3 ratio is `~1/3 = sqrt(1/9)`.

**Why this matters at training time.** Wrong `fan_in` (using `IC` alone for a `3x3` conv) gives weights `3x` too large — pre-activations explode by `~3x` per layer, saturating ReLUs and killing gradients in the first few backward passes. The empirical-std test catches this.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()